# Exploratory Data Analysis

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [3]:
raw_data = pd.read_csv('complete_data.csv')
print(f"Liczba obserwacji: {raw_data.shape[0]}")
print(f"Liczba zmiennych: {raw_data.shape[1]}")
raw_data.head()

Liczba obserwacji: 11998
Liczba zmiennych: 8


,Marka,Model,Cena,Waluta,Paliwo,Przebieg,Rocznik,Skrzynia biegów
0,Alfa Romeo,Alfa Romeo 147,3000,PLN,Benzyna,159000,2008,Manualna
1,Volvo,Volvo XC 60,129999,PLN,Benzyna,127000,2018,Automatyczna
2,Ford,Ford B-MAX,22900,PLN,Benzyna,161000,2016,Manualna
3,Jaguar,Jaguar I-Pace EV400 AWD SE,69900,PLN,Elektryczny,135125,2019,Automatyczna
4,Cupra,Cupra Formentor VZ 2.0 TSI 4Drive DSG,119999,PLN,Benzyna,81800,2020,Automatyczna


## Pojemność silnika- dodanie zmiennej

In [15]:
df = raw_data.copy()
df['Pojemność silnika'] = pd.to_numeric(
    df['Model'].str.extract(r'\b([1-5]\.\d)\b', expand=False), 
    errors='coerce'
)
print(df[['Model', 'Pojemność silnika']].head(15))

                                    Model  Pojemność silnika
0                          Alfa Romeo 147                NaN
1                             Volvo XC 60                NaN
2                              Ford B-MAX                NaN
3              Jaguar I-Pace EV400 AWD SE                NaN
4   Cupra Formentor VZ 2.0 TSI 4Drive DSG                2.0
5       Hyundai Kona 1.0 T-GDI Advantage+                1.0
6                             Denza Z9 GT                NaN
7                  Fiat Tipo 1.4 16V More                1.4
8                Toyota RAV4 2.0 D-4D 4x4                2.0
9        Renault Megane 1.9 dCi Dynamique                1.9
10   Jeep Compass 2.0 MJD Limited 4WD S&S                2.0
11                          Opel Insignia                NaN
12                         Renault Trafic                NaN
13                          Volkswagen CC                NaN
14        Ford Focus 1.0 EcoBoost ST-Line                1.0


In [12]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11998 entries, 0 to 11997
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Marka            11998 non-null  object
 1   Model            11998 non-null  object
 2   Cena             11998 non-null  int64 
 3   Waluta           11998 non-null  object
 4   Paliwo           11998 non-null  object
 5   Przebieg         11998 non-null  int64 
 6   Rocznik          11998 non-null  int64 
 7   Skrzynia biegów  11998 non-null  object
dtypes: int64(3), object(5)
memory usage: 750.0+ KB


## Unikalne wartości w zbiorze danych

In [ ]:
for column in df.columns:
    unique_values = df[column].nunique()
    print(f"Liczba unikalnych wartości w kolumnie '{column}': {unique_values}")

Liczba unikalnych wartości w kolumnie 'Marka': 99
Liczba unikalnych wartości w kolumnie 'Model': 6619
Liczba unikalnych wartości w kolumnie 'Cena': 2371
Liczba unikalnych wartości w kolumnie 'Waluta': 2
Liczba unikalnych wartości w kolumnie 'Paliwo': 8
Liczba unikalnych wartości w kolumnie 'Przebieg': 6047
Liczba unikalnych wartości w kolumnie 'Rocznik': 62
Liczba unikalnych wartości w kolumnie 'Skrzynia biegów': 2
Liczba unikalnych wartości w kolumnie 'Pojemność silnika': 30
Liczba unikalnych wartości w kolumnie 'Pojemność_znana': 2


In [34]:
df['Waluta'].unique()

array(['PLN', 'EUR'], dtype=object)

### Konwersja walut

In [36]:
oferty_euro = df[df['Waluta'] == 'EUR']
print(f"Zidentyfikowano {len(oferty_euro)} ofert w Euro.")

Kurs_EUR_PLN = 4.32

mask_euro = df['Waluta'] == 'EUR'
df.loc[mask_euro, 'Cena'] = df.loc[mask_euro, 'Cena'] * Kurs_EUR_PLN
df.loc[mask_euro, 'Waluta'] = 'PLN'

Zidentyfikowano 0 ofert w Euro.


## Brakujące dane

In [ ]:
missing = df.isnull().sum()
missing_percentage = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Brakujące wartości': missing, 'Procent brakujących wartości': missing_percentage})
missing

Marka                   0
Model                   0
Cena                    0
Waluta                  0
Paliwo                  0
Przebieg                0
Rocznik                 0
Skrzynia biegów         0
Pojemność silnika    6861
dtype: int64

Ponad połowa obserwacji nie posiada danych na temat pojemności silnika, co stanowi problem, który warto rozwiązać

### Pojemność silnika- problem braku danych w części obserwacji
Ekstrakcja wartości z tekstu ogłoszenia za pomocą wyrażenia regularnego oraz utworzenie binarnej zmiennej wskaźnikowej (Pojemność_znana). Brakujące wartości uzupełniono hierarchicznie (mediana dla grupy Marka-Model, następnie Marka, a na końcu globalna mediana), co pozwala zachować pełną próbę danych i uniknąć ich odrzucenia w dalszej analizie.

In [ ]:
df['Pojemność silnika'] = pd.to_numeric(
    df['Model'].str.extract(r'\b([1-5]\.\d)\b', expand=False), 
    errors='coerce'
)
df['Pojemność_znana'] = df['Pojemność silnika'].notnull().astype(int)
df['Pojemność silnika'] = df.groupby(['Marka', 'Model'])['Pojemność silnika'].transform(lambda x: x.fillna(x.median()))
df['Pojemność silnika'] = df.groupby('Marka')['Pojemność silnika'].transform(lambda x: x.fillna(x.median()))
df['Pojemność silnika'] = df['Pojemność silnika'].fillna(df['Pojemność silnika'].median())
print(df[['Model', 'Pojemność silnika', 'Pojemność_znana']].head(10))

                                   Model  Pojemność silnika  Pojemność_znana
0                         Alfa Romeo 147                2.0                0
1                            Volvo XC 60                2.4                0
2                             Ford B-MAX                1.6                0
3             Jaguar I-Pace EV400 AWD SE                2.0                0
4  Cupra Formentor VZ 2.0 TSI 4Drive DSG                2.0                1
5      Hyundai Kona 1.0 T-GDI Advantage+                1.0                1
6                            Denza Z9 GT                1.6                0
7                 Fiat Tipo 1.4 16V More                1.4                1
8               Toyota RAV4 2.0 D-4D 4x4                2.0                1
9       Renault Megane 1.9 dCi Dynamique                1.9                1


In [ ]:
missing_after_imputation = df['Pojemność silnika'].isnull().sum()
missing_after_imputation

np.int64(0)

In [37]:
df.to_csv('cars_data.csv', index=False, encoding='utf-8-sig', na_rep='NULL')

## Analiza
1. Zrozumienie struktury cen samochodów osobowych na rynku wtórnym i pierwotnym w okolicach Krakowa oraz przygotowanie bazy pod model regresji liniowej
2. Docelowy plan -> przewidywanie rynkowej ceny samochodu na podstawie jego kluczowych parametrów technicznych i eksploatacyjnych

In [39]:
data = pd.read_csv('cars_data.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11998 entries, 0 to 11997
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Marka              11998 non-null  object 
 1   Model              11998 non-null  object 
 2   Cena               11998 non-null  int64  
 3   Waluta             11998 non-null  object 
 4   Paliwo             11998 non-null  object 
 5   Przebieg           11998 non-null  int64  
 6   Rocznik            11998 non-null  int64  
 7   Skrzynia biegów    11998 non-null  object 
 8   Pojemność silnika  11998 non-null  float64
 9   Pojemność_znana    11998 non-null  int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 937.5+ KB


In [40]:
print(data.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

                     count           mean            std     min      25%  \
Cena               11998.0   77316.924654  100117.192850  1350.0  22900.0   
Przebieg           11998.0  142182.882230  100359.241509     0.0  66600.0   
Rocznik            11998.0    2016.261627       7.033279  1951.0   2012.0   
Pojemność silnika  11998.0       1.791349       0.585278     1.0      1.5   
Pojemność_znana    11998.0       0.428155       0.494832     0.0      0.0   

                        50%       75%        max  
Cena                47000.0   95000.0  1599000.0  
Przebieg           139850.0  202185.0  2550002.0  
Rocznik              2017.0    2022.0     2026.0  
Pojemność silnika       1.6       2.0        5.7  
Pojemność_znana         0.0       1.0        1.0  
